In [1]:
import pandas as pd
import gc
from mnp.ingestion.loader import load_endes
from IPython.display import display


# Carga de historial desde 2007 para RECH0
history = load_endes(
        year=range(2007, 2025),
        module='household',
        record='household_characteristics',
        meta=True
    )

frames = []
latest_year_labels = {} # labels del ultimo año
col_labels_hist = {} # labels de columna por año
value_labels_history = {} # labels de valores por año

# Extraemos y liberamos memoria progresivamente usando pop()
for year in sorted(list(history.keys())):
    year_df, year_meta = history.pop(year)
    frames.append(year_df.assign(year=year))
    latest_year_labels.update(year_meta.column_names_to_labels)
    col_labels_hist[year] = year_meta.column_names_to_labels
    value_labels_history[year] = year_meta.variable_value_labels

df_rech0 = pd.concat(frames, ignore_index=True)
del frames
gc.collect() # Forzar limpieza de memoria

print(f"Registros RECH0 totales: {len(df_rech0)}")
print(f"Columnas RECH0 totales: {df_rech0.shape[1]}")

2026-06-01 22:11:56.436 | INFO     | mnp.config:<module>:11 - PROJ_ROOT path is: /Users/abelguevarah/Desktop/invs/malnutrition-research


2026-06-01 22:11:56.529 | INFO     | mnp.ingestion.loader:load_endes:375 - ✓ Cargando registro: RECH0.SAV desde Modulo64 [Alias: household_characteristics]
2026-06-01 22:11:56.826 | INFO     | mnp.ingestion.loader:load_endes:375 - ✓ Cargando registro: RECH0.sav desde Modulo64 [Alias: household_characteristics]
2026-06-01 22:11:57.123 | INFO     | mnp.ingestion.loader:load_endes:375 - ✓ Cargando registro: RECH0.SAV desde Modulo64 [Alias: household_characteristics]
2026-06-01 22:11:57.308 | INFO     | mnp.ingestion.loader:load_endes:375 - ✓ Cargando registro: RECH0.SAV desde Modulo64 [Alias: household_characteristics]
2026-06-01 22:11:57.518 | INFO     | mnp.ingestion.loader:load_endes:375 - ✓ Cargando registro: RECH0.sav desde Modulo64 [Alias: household_characteristics]
2026-06-01 22:11:57.723 | INFO     | mnp.ingestion.loader:load_endes:375 - ✓ Cargando registro: RECH0.sav desde Modulo64 [Alias: household_characteristics]
2026-06-01 22:11:57.949 | INFO     | mnp.ingestion.loader:load_e

In [2]:
# 1. Analizador Global de "Variable Label Drift" (Mutación de Nombres Descriptivos)
from collections import defaultdict
from mnp.ingestion.loader import format_year_ranges

def analyze_drift(col_labels_hist):
    """Analiza la mutación de nombres descriptivos a lo largo de los años."""
    drift_report = defaultdict(lambda: defaultdict(list))
    all_cols = {c for year_columns in col_labels_hist.values() for c in year_columns}

    for col_name in sorted(all_cols):
        for year, year_columns in col_labels_hist.items():
            if col_name in year_columns:
                label_val = year_columns[col_name]
                if label_val is None or str(label_val).strip() == "" or str(label_val).strip() == "None" or str(label_val).strip().upper() == col_name.upper():
                    continue
                description = str(label_val).strip()
                drift_report[col_name][description].append(year)

    mutated_labels = {k: v for k, v in drift_report.items() if len(v) > 1}
    stable_labels = {k: v for k, v in drift_report.items() if len(v) == 1}
    
    return all_cols, stable_labels, mutated_labels

def print_drift_report(all_cols, stable_labels, mutated_labels):
    """Imprime el reporte global de mutación de descripciones."""
    print("REPORTE GLOBAL DE MUTACIÓN DE DESCRIPCIONES (2007 - 2024)\n")
    print(f"Total de columnas físicas históricas detectadas: {len(all_cols)}\n")

    # Compute global max padding length for years across both stable and mutated labels
    all_versions = []
    for label_dict in [stable_labels, mutated_labels]:
        for versions in label_dict.values():
            all_versions.extend(versions.values())
    global_max_len = max([len(format_year_ranges(a)) for a in all_versions] + [0])

    for title, label_dict in [("Descripciones estables", stable_labels), ("Descripciones mutadas", mutated_labels)]:
        print(f"{title}: {len(label_dict)}")
        for var_name, versions in label_dict.items():
            print(f"{var_name} (Apariciones: {sum(len(y) for y in versions.values())} años)")
            for desc_text, years_list in versions.items():
                years_str = format_year_ranges(years_list)
                print(f"  - Años {years_str:<{global_max_len}} : {desc_text}")
        print("\n")

def audit_missing_desc(col_labels_hist, all_cols):
    """Realiza la auditoría de descripciones faltantes e imprime los resultados."""
    print("AUDITORÍA DE DESCRIPCIONES FALTANTES\n")
    print("Variables existentes sin texto descriptivo:\n")
    
    alerts = []
    for col_name in sorted(all_cols):
        years_present = []
        years_without_desc = []
        for year, year_columns in col_labels_hist.items():
            if col_name in year_columns:
                years_present.append(year)
                label_val = year_columns[col_name]
                if label_val is None or str(label_val).strip() == "" or str(label_val).strip() == "None" or str(label_val).strip().upper() == col_name.upper():
                    years_without_desc.append(year)
                    
        if years_without_desc:
            alerts.append((col_name, format_year_ranges(years_without_desc), format_year_ranges(years_present)))

    if not alerts:
        print("No se detectaron descripciones faltantes.")
    else:
        max_col_len = max(len(col) for col, _, _ in alerts)
        max_missing_len = max(len(missing) for _, missing, _ in alerts)
        
        for col_name, missing, present in alerts:
            print(f"{col_name:<{max_col_len}} | Sin descripción en: {missing:<{max_missing_len}} (Existió en: {present})")
            
        print(f"\nTotal de variables con descripciones faltantes: {len(alerts)}\n")
# --- Ejecución Principal ---
all_cols, stable_labels, mutated_labels = analyze_drift(col_labels_hist)
print_drift_report(all_cols, stable_labels, mutated_labels)
audit_missing_desc(col_labels_hist, all_cols)


REPORTE GLOBAL DE MUTACIÓN DE DESCRIPCIONES (2007 - 2024)

Total de columnas físicas históricas detectadas: 61

Descripciones estables: 15
HV005A (Apariciones: 1 años)
  - Años 2008                                  : Sample weight Departamental
HV005X (Apariciones: 1 años)
  - Años 2015                                  : Factor de ponderación niño menor de 5 años 2015
ID1 (Apariciones: 13 años)
  - Años 2010-2016, 2019-2024                  : Año
LATITUDY (Apariciones: 1 años)
  - Años 2022                                  : Latitud del conglomerado
LONGITUDX (Apariciones: 1 años)
  - Años 2022                                  : Longitud del conglomerado
NCONGLOME (Apariciones: 5 años)
  - Años 2019-2023                             : Número de Conglomerado (proveniente del marco)
NCONGLOME1 (Apariciones: 1 años)
  - Años 2024                                  : Número de Conglomerado (proveniente del marco)
UBIGEO (Apariciones: 6 años)
  - Años 2019-2024                             : Có